# Unione ed elaborazione dati Sanremo (1951–2023)

**Esercitazione finale – M5 Looker Studio**

Obiettivo: unire le tre tabelle fornite (Festival, Classifica, Spotify) in un unico dataset pulito e arricchito, pronto per essere caricato su Google Sheets e collegato a Looker Studio.

Tema scelto: **la storia del Festival e l'eredità delle canzoni su Spotify oggi.**

Le tre fonti:
- `dati-festival-sanremo-1951-2023.xlsx` — una riga per edizione (sede, presentatore, partecipanti, vincitore)
- `dati-classifica-sanremo-1951-2023.xlsx` — una riga per canzone in gara, con posizione finale
- `dati-canzoni-spotify-sanremo-1951-2023.xlsx` — dati Spotify per singola canzone (popolarità attuale, durata)


In [1]:
import pandas as pd
import re

SRC = 'Data/'  # cartella con i 3 file originali

df_fest = pd.read_excel(SRC + 'dati-festival-sanremo-1951-2023.xlsx')
df_class = pd.read_excel(SRC + 'dati-classifica-sanremo-1951-2023.xlsx').drop(columns=['Unnamed: 0'])
df_spot = pd.read_excel(SRC + 'dati-canzoni-spotify-sanremo-1951-2023.xlsx')

df_class['anno'] = df_class['anno'].astype(int)

print('Festival:', df_fest.shape)
print('Classifica:', df_class.shape)
print('Spotify:', df_spot.shape)


Festival: (73, 6)
Classifica: (1663, 5)
Spotify: (1582, 6)


## 1. Pulizia della colonna `Posizione`

La colonna mescola numeri e testo: `"1º"`, `"3º (ex aequo)"`, `"F"` (fuori classifica/finalista), `"NF"` (non finalista). 
La scomponiamo in:
- `Posizione_num`: il numero puro (vuoto se non applicabile)
- `Posizione_stato`: categoria testuale
- `Vincitrice_brano` / `Podio`: flag booleani derivati


In [2]:
def parse_pos(p):
    if pd.isna(p):
        return (None, 'Non classificata')
    s = str(p).strip()
    if s == 'F':
        return (None, 'Finalista (fuori classifica)')
    if s == 'NF':
        return (None, 'Non finalista')
    m = re.match(r'^(\d+)', s)
    if m:
        return (int(m.group(1)), 'Classificata')
    return (None, 'Non classificata')

parsed = df_class['Posizione'].apply(parse_pos)
df_class['Posizione_num'] = [x[0] for x in parsed]
df_class['Posizione_stato'] = [x[1] for x in parsed]
df_class['Vincitrice_brano'] = df_class['Posizione_num'] == 1
df_class['Podio'] = df_class['Posizione_num'].isin([1, 2, 3])

df_class['Posizione_stato'].value_counts()


Posizione_stato
Classificata                    1057
Non finalista                    433
Finalista (fuori classifica)     106
Non classificata                  67
Name: count, dtype: int64

## 2. Pulizia nomi e calcolo Decennio (tabella Festival)

Rimuoviamo i suffissi tra parentesi (es. `"Amadeus (conduttore televisivo)"` → `"Amadeus"`) e calcoliamo il decennio di ogni edizione, utile per aggregare i grafici in Looker Studio.


In [3]:
def strip_paren(s):
    if pd.isna(s):
        return s
    return re.sub(r'\s*\([^)]*\)', '', str(s)).strip()

df_fest_clean = df_fest.copy()
df_fest_clean['Presentatore_pulito'] = df_fest_clean['Presentatore'].apply(strip_paren)
df_fest_clean['Vincitore_pulito'] = df_fest_clean['Vincitore'].apply(strip_paren)

def decennio(anno):
    d = (anno // 10) * 10
    return f"Anni '{str(d)[-2:]}"

df_fest_clean['Decennio'] = df_fest_clean['Anno'].apply(decennio)

df_fest_clean[['Anno','Decennio','Presentatore_pulito','Vincitore_pulito']].head()


,Anno,Decennio,Presentatore_pulito,Vincitore_pulito
0,1951,Anni '50,Nunzio Filogamo,Nilla Pizzi
1,1952,Anni '50,Nunzio Filogamo,Nilla Pizzi
2,1953,Anni '50,Nunzio Filogamo,Carla Boni
3,1954,Anni '50,Nunzio Filogamo,Gino Latilla
4,1955,Anni '50,Armando Pizzo,Claudio Villa


## 3. Unione Classifica + Festival (contesto edizione)

Portiamo in ogni riga-canzone il contesto della sua edizione (sede, presentatore, partecipanti), unendo su `anno`.


In [4]:
df = df_class.merge(
    df_fest_clean[['Anno', 'Periodo', 'Sede', 'Presentatore_pulito', 'Partecipanti', 'Vincitore_pulito', 'Decennio']],
    left_on='anno', right_on='Anno', how='left'
).drop(columns=['Anno'])

df['Interprete_pulito'] = df['Interprete'].apply(strip_paren)
df.shape


(1663, 16)

## 4. Unione con i dati Spotify

Qui la parte più delicata: il join si fa sul titolo del brano (`Canzone`), ma **alcuni titoli si ripetono** in edizioni diverse (es. *"Vorrei"* compare 3 volte). Un merge semplice su `Canzone` genera righe duplicate.

**Soluzione:** dopo il merge, per ogni riga originale della classifica, se ci sono più candidati Spotify con lo stesso titolo, teniamo quello il cui campo `Artista` condivide più parole con l'`Interprete` originale (match per token).


In [5]:
def name_tokens(s):
    if pd.isna(s):
        return set()
    s = re.sub(r'\([^)]*\)', '', str(s))
    s = re.sub(r'[^\w\s]', ' ', s.lower())
    return set(t for t in s.split() if len(t) > 2)

merged = df.merge(df_spot, on='Canzone', how='left', suffixes=('', '_spot'))
print('Righe dopo merge grezzo (con duplicati da omonimi):', len(merged))

def score_row(row):
    a = name_tokens(row['Interprete'])
    b = name_tokens(row.get('Artista'))
    if not a or not b:
        return 0
    return len(a & b)

merged['_score'] = merged.apply(score_row, axis=1)
merged['_rowkey'] = (merged['anno'].astype(str) + '|' + merged['Canzone'] + '|' +
                      merged['Interprete'].astype(str) + '|' + merged['Posizione'].astype(str))

merged = (merged.sort_values('_score', ascending=False)
                 .drop_duplicates(subset='_rowkey', keep='first')
                 .drop(columns=['_score', '_rowkey']))

print('Righe dopo deduplicazione (una per canzone/edizione):', len(merged))


Righe dopo merge grezzo (con duplicati da omonimi): 1745
Righe dopo deduplicazione (una per canzone/edizione): 1654


## 5. Campi calcolati per lo storytelling

- `Presente_su_Spotify`: il brano è stato trovato su Spotify?
- `Classe_popolarita`: fascia di popolarità (0-20 ... 81-100), utile per grafici a torta/pivot
- `Anni_dal_Festival_a_ultima_pubblicazione`: quanto tempo è passato tra l'edizione e l'ultima ripubblicazione del brano — un valore alto indica una "seconda vita" del brano


In [6]:
merged['Presente_su_Spotify'] = merged['Popolarità'].notna()

def classe_pop(p):
    if pd.isna(p):
        return 'Non disponibile'
    if p <= 20: return '1. Bassa (0-20)'
    if p <= 40: return '2. Medio-bassa (21-40)'
    if p <= 60: return '3. Media (41-60)'
    if p <= 80: return '4. Alta (61-80)'
    return '5. Molto alta (81-100)'

merged['Classe_popolarita'] = merged['Popolarità'].apply(classe_pop)
merged['Anni_da_uscita_a_oggi'] = 2024 - merged['Anno_ultimo_rilascio']
merged['Anni_dal_Festival_a_ultima_pubblicazione'] = merged['Anno_ultimo_rilascio'] - merged['anno']

merged = merged.sort_values(['anno', 'Posizione_num'], na_position='last').reset_index(drop=True)

print(f"Copertura match Spotify: {merged['Presente_su_Spotify'].mean()*100:.1f}%")


Copertura match Spotify: 94.3%


## 6. Tabella finale

Selezioniamo e rinominiamo le colonne per la tabella `Dati_Uniti` che useremo in Looker Studio (una riga = una canzone in gara in una edizione).


In [7]:
final_cols = [
    'anno', 'Decennio', 'Periodo', 'Sede', 'Presentatore_pulito', 'Partecipanti',
    'Posizione', 'Posizione_num', 'Posizione_stato', 'Podio', 'Vincitrice_brano',
    'Interprete_pulito', 'Canzone', 'Autori', 'Vincitore_pulito',
    'Artista', 'Anno_ultimo_rilascio', 'Anni_dal_Festival_a_ultima_pubblicazione',
    'Durata (min)', 'Popolarità', 'Classe_popolarita', 'Presente_su_Spotify',
    'URL immagine album'
]
final = merged[final_cols].rename(columns={
    'anno': 'Anno',
    'Presentatore_pulito': 'Presentatore',
    'Interprete_pulito': 'Interprete',
    'Vincitore_pulito': 'Vincitore_edizione',
    'Durata (min)': 'Durata_min',
    'Popolarità': 'Popolarita_Spotify'
})

print(final.shape)
final.head()


(1654, 23)


,Anno,Decennio,Periodo,Sede,Presentatore,Partecipanti,Posizione,Posizione_num,Posizione_stato,Podio,...,Autori,Vincitore_edizione,Artista,Anno_ultimo_rilascio,Anni_dal_Festival_a_ultima_pubblicazione,Durata_min,Popolarita_Spotify,Classe_popolarita,Presente_su_Spotify,URL immagine album
0,1951,Anni '50,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,3,1º,1.0,Classificata,True,...,Gian Carlo Testoni e Mario Panzeri,Nilla Pizzi,NaN,NaN,NaN,NaN,NaN,Non disponibile,False,NaN
1,1951,Anni '50,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,3,2º,2.0,Classificata,True,...,Biri,Nilla Pizzi,NaN,NaN,NaN,NaN,NaN,Non disponibile,False,NaN
2,1951,Anni '50,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,3,3º,3.0,Classificata,True,...,Walter Colì,Nilla Pizzi,NaN,NaN,NaN,NaN,NaN,Non disponibile,False,NaN
3,1951,Anni '50,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,3,F,NaN,Finalista (fuori classifica),False,...,Aldo Valleroni e Da Rovere,Nilla Pizzi,NaN,NaN,NaN,NaN,NaN,Non disponibile,False,NaN
4,1951,Anni '50,29-31 gennaio,Casinò di Sanremo,Nunzio Filogamo,3,F,NaN,Finalista (fuori classifica),False,...,Astro Mari,Nilla Pizzi,NaN,NaN,NaN,NaN,NaN,Non disponibile,False,NaN


## 7. Verifica: il messaggio chiave dello storytelling

Confrontiamo la popolarità media Spotify dei brani **vincitori** rispetto a quelli **non vincitori** — è il numero su cui costruiamo il racconto del report.


In [8]:
confronto = final[final['Presente_su_Spotify']].groupby('Vincitrice_brano')['Popolarita_Spotify'].mean().round(1)
confronto.index = ['Non vincitori', 'Vincitori']
confronto


Non vincitori    28.9
Vincitori        51.6
Name: Popolarita_Spotify, dtype: float64

## 8. Esportazione

Salviamo il dataset unito (`Dati_Uniti`) e la tabella edizioni (`Edizioni_Festival`) in un unico file Excel, pronto per essere caricato su Google Sheets e collegato a Looker Studio.


In [9]:
fest_out = df_fest_clean[['Anno','Decennio','Periodo','Sede','Presentatore_pulito','Partecipanti','Vincitore_pulito']].rename(
    columns={'Presentatore_pulito':'Presentatore','Vincitore_pulito':'Vincitore'}
)

with pd.ExcelWriter(SRC + 'dati-sanremo-uniti-python.xlsx', engine='openpyxl') as writer:
    final.to_excel(writer, sheet_name='Dati_Uniti', index=False)
    fest_out.to_excel(writer, sheet_name='Edizioni_Festival', index=False)

print('File esportato: dati-sanremo-uniti-python.xlsx')


File esportato: dati-sanremo-uniti-python.xlsx


## 9. Classificazione di genere dei vincitori

Per il tema scelto (dal palco a Spotify, con un focus sulla rappresentanza di genere tra i vincitori)
serve un campo che il dataset originale non contiene: il genere dell'artista o del gruppo vincitore
di ogni edizione.

Le edizioni sono solo 73, quindi la classificazione è stata fatta a mano, edizione per edizione,
usando queste categorie:

- `Uomo` / `Donna`: artista solista
- `Gruppo maschile` / `Gruppo femminile`: gruppo/band composto da un solo genere
- `Gruppo misto`: gruppo o duo con partecipanti di entrambi i generi

**Limite noto:** nel 2022 "Brividi" ha vinto in coppia (Mahmood e Blanco), ma il dataset originale
riporta solo "Mahmood" come `Vincitore`. La riga è classificata come `Uomo`, con la spiegazione
salvata nella colonna `Note_genere`.

In [10]:
genere_vincitore = {
    1951: "Donna", 1952: "Donna", 1953: "Donna", 1954: "Uomo", 1955: "Uomo",
    1956: "Donna", 1957: "Uomo", 1958: "Uomo", 1959: "Uomo", 1960: "Uomo",
    1961: "Donna", 1962: "Uomo", 1963: "Uomo", 1964: "Donna", 1965: "Uomo",
    1966: "Uomo", 1967: "Uomo", 1968: "Uomo", 1969: "Uomo", 1970: "Uomo",
    1971: "Donna", 1972: "Uomo", 1973: "Uomo", 1974: "Donna", 1975: "Donna",
    1976: "Uomo", 1977: "Gruppo maschile", 1978: "Gruppo misto", 1979: "Uomo",
    1980: "Uomo", 1981: "Donna", 1982: "Uomo", 1983: "Donna", 1984: "Uomo",
    1985: "Gruppo misto", 1986: "Uomo", 1987: "Uomo", 1988: "Uomo", 1989: "Donna",
    1990: "Gruppo maschile", 1991: "Uomo", 1992: "Uomo", 1993: "Uomo", 1994: "Uomo",
    1995: "Donna", 1996: "Uomo", 1997: "Gruppo misto", 1998: "Donna", 1999: "Donna",
    2000: "Gruppo maschile", 2001: "Donna", 2002: "Gruppo misto", 2003: "Donna",
    2004: "Uomo", 2005: "Uomo", 2006: "Uomo", 2007: "Uomo", 2008: "Uomo",
    2009: "Uomo", 2010: "Uomo", 2011: "Uomo", 2012: "Donna", 2013: "Uomo",
    2014: "Donna", 2015: "Gruppo maschile", 2016: "Gruppo maschile", 2017: "Uomo",
    2018: "Uomo", 2019: "Uomo", 2020: "Uomo", 2021: "Gruppo misto", 2022: "Uomo",
    2023: "Uomo",
}

note_genere = {
    2022: "Vittoria in coppia con Blanco (\"Brividi\"); il dataset originale riporta solo Mahmood come Vincitore"
}

fest_out["Genere_vincitore"] = fest_out["Anno"].map(genere_vincitore)
fest_out["Note_genere"] = fest_out["Anno"].map(note_genere)

fest_out[["Anno", "Vincitore", "Genere_vincitore"]].tail(10)

,Anno,Vincitore,Genere_vincitore
63,2014,Arisa,Donna
64,2015,Il Volo,Gruppo maschile
65,2016,Stadio,Gruppo maschile
66,2017,Francesco Gabbani,Uomo
67,2018,Ermal Meta,Uomo
68,2019,Mahmood,Uomo
69,2020,Diodato,Uomo
70,2021,Måneskin,Gruppo misto
71,2022,Mahmood,Uomo
72,2023,Marco Mengoni,Uomo


## 10. Esportazione del dataset completo

Riscriviamo il file Excel aggiungendo le due nuove colonne al foglio `Edizioni_Festival` e un foglio
`Nota_Genere` con il criterio di classificazione, da citare nel report per trasparenza metodologica.

In [11]:
nota_genere_rows = [
    ("Nota metodologica - Genere_vincitore", ""),
    ("", ""),
    ("Classificazione", "Criterio"),
    ("Uomo / Donna", "Artista solista, genere anagrafico noto"),
    ("Gruppo maschile / femminile", "Gruppo/band composto da soli uomini o sole donne"),
    ("Gruppo misto", "Gruppo/band o duo con partecipanti di entrambi i generi"),
    ("", ""),
    ("Limite noto", "Nel 2022 \'Brividi\' ha vinto in coppia (Mahmood e Blanco), ma il dataset "
                    "originale riporta solo \'Mahmood\' come Vincitore. Classificato come Uomo, "
                    "con nota nella colonna Note_genere."),
]
df_nota_genere = pd.DataFrame(nota_genere_rows, columns=["Metodologia ed elaborazione dati", ""])

with pd.ExcelWriter(SRC + "dati-sanremo-uniti-completo.xlsx", engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="Dati_Uniti", index=False)
    fest_out.to_excel(writer, sheet_name="Edizioni_Festival", index=False)
    df_nota_genere.to_excel(writer, sheet_name="Nota_Genere", index=False, header=False)

print("File esportato: dati-sanremo-uniti-completo.xlsx")

File esportato: dati-sanremo-uniti-completo.xlsx
